In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Set seeds
tf.random.set_seed(42)
np.random.seed(42)

In [ ]:
from utils.data import load_and_merge_data

merged_ecg, merged_ppg = load_and_merge_data('/root/tmp/PulseGUARD/data_gan_3s')

print(merged_ecg.shape, merged_ppg.shape)

In [ ]:
# Data loading and preprocessing
ppg_data = merged_ppg.astype(np.float32)
ecg_data = merged_ecg.astype(np.float32)

# Normalize to [-1, 1]
ppg_data = 2 * (ppg_data - np.min(ppg_data)) / (np.max(ppg_data) - np.min(ppg_data)) - 1
ecg_data = 2 * (ecg_data - np.min(ecg_data)) / (np.max(ecg_data) - np.min(ecg_data)) - 1

# Expand dimensions
ppg_data = np.expand_dims(ppg_data, axis=-1)
ecg_data = np.expand_dims(ecg_data, axis=-1)

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(ppg_data, ecg_data, test_size=0.2, random_state=42)
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(1000).batch(32)
test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(1)

In [ ]:
from utils.model import build_enhanced_discriminator,build_unet_lstm_generator
from utils.train import train

# Initialize and train model
seq_len = ppg_data.shape[1]
generator = build_unet_lstm_generator(seq_len)
discriminator = build_enhanced_discriminator(seq_len)

# Debug test dataset
for ppg, ecg in test_dataset.take(1):
    print(f"Test batch shapes: ppg {ppg.shape}, ecg {ecg.shape}")

mse_filepath = '/root/tmp/PulseGUARD/model/best_generator_mse.h5'
ppc_filepath = '/root/tmp/PulseGUARD/model/best_generator_pcc.h5'

train(train_dataset,test_dataset, epochs=500, generator=generator, discriminator=discriminator, mse_filepath=mse_filepath , ppc_filepath=ppc_filepath)

In [ ]:
# Visualized prediction results
import random
plt.figure(figsize=(15, 10))
for i in range(5):
    idx = random.randint(0, len(X_test)-1)
    ppg_sample = X_test[idx:idx+1]
    true_ecg = y_test[idx].flatten()
    pred_ecg = generator(ppg_sample, training=False).numpy().flatten()
    

    plt.subplot(5, 1, i+1)
    plt.plot(true_ecg, label='True ECG')
    plt.plot(pred_ecg, label='Pred ECG')
    plt.legend()
    plt.title(f'Sample {idx}')
plt.tight_layout()
plt.show()